# مرحله ۱: تست دود Matcha-TTS در Colab

هدف این مرحله فقط یک چیز است: مطمئن شویم زنجیره کامل کار می‌کند؛ یعنی متن → مدل Matcha-TTS → مل‌اسپکتروگرام → وکودر HiFi-GAN → فایل صوتی.

در این مرحله هیچ آموزشی انجام نمی‌شود و DriftTTS را از صفر نمی‌سازیم. فقط با مدل آماده Matcha-TTS یک جمله را با ۱۰ گام (steps=10) تولید می‌کنیم. عدد ۱۰ همان پیش‌فرض خود Matcha-TTS است و در مقاله DriftTTS هم مرجع چندمرحله‌ای با ۱۰ گام بود.

ترتیب اجرا: سلول‌ها را از بالا به پایین، یکی‌یکی اجرا کن. اگر سلولی خطا داد، قبل از اجرای سلول بعد، همان خطا را نگه دار و بفرست.

## قدم ۰: بررسی GPU

اول باید بدانیم Colab واقعا GPU داده است یا نه. اگر GPU نباشد، مدل روی CPU هم کار می‌کند ولی خیلی کندتر می‌شود. خروجی مورد انتظار: نام یک کارت مثل T4 و عبارت True برای cuda.

In [ ]:
import sys
import torch

print('Python:', sys.version)
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## قدم ۱: نصب espeak-ng

Matcha-TTS برای تبدیل متن به واج (phoneme) به espeak نیاز دارد. اگر این بسته نباشد، معمولا خطای «espeak not installed» می‌گیری. این سلول آن را نصب می‌کند.

In [ ]:
!apt-get update -qq
!apt-get install -y espeak-ng
!which espeak-ng || which espeak || echo 'espeak NOT FOUND'

## قدم ۲: نصب Matcha-TTS

با این دستور، بسته رسمی matcha-tts از PyPI نصب می‌شود. در اولین اجرای سنتز، مدل آماده matcha_ljspeech (تک‌گوینده، LJSpeech) و وکودر hifigan_T2_v1 به‌صورت خودکار دانلود می‌شوند. این قدم چند دقیقه طول می‌کشد؛ صبر کن تا سلول کامل شود.

In [ ]:
!pip install matcha-tts

## قدم ۳: یک جمله آزمایشی با ۱۰ گام

جمله ثابت مرحله ۱:

> Hello, this is a test of Matcha text to speech with ten steps.

پارامتر steps همان NFE است: تعداد دفعاتی که دیکدر در حل ODE صدا زده می‌شود. در مرحله بعد، همین جمله را با steps برابر ۱، ۲، ۴، ۱۰ و ۲۵ تکرار می‌کنیم و سرعت و کیفیت را مقایسه می‌کنیم. خروجی این سلول باید utterance_001.wav باشد.

In [ ]:
import pathlib
import subprocess
import time

out_dir = pathlib.Path('step1_outputs')
out_dir.mkdir(exist_ok=True)
text = 'Hello, this is a test of Matcha text to speech with ten steps.'

t0 = time.time()
result = subprocess.run(
    ['matcha-tts', '--text', text, '--steps', '10', '--output_folder', str(out_dir)],
    capture_output=True, text=True
)
print('return code:', result.returncode)
print('wall time (s):', round(time.time() - t0, 1))
print('--- stdout (tail) ---')
print(result.stdout[-2000:])
print('--- stderr (tail) ---')
print(result.stderr[-2000:])
print('files:', sorted(str(p) for p in out_dir.glob('*')))

## قدم ۴: گوش دادن و بررسی فایل

اگر قدم ۳ موفق بوده باشد، این سلول نرخ نمونه‌برداری، مدت صدا و حجم فایل را چاپ می‌کند و خود صدا را هم پخش می‌کند. اگر صدا قابل فهم و شبیه یک جمله انگلیسی طبیعی بود، مرحله ۱ قبول است.

In [ ]:
import contextlib
import os
import wave

from IPython.display import Audio, display

wav_path = 'step1_outputs/utterance_001.wav'
assert os.path.exists(wav_path), f'Not found: {wav_path}. Run step 3 again and check its error.'
with contextlib.closing(wave.open(wav_path)) as w:
    duration = w.getnframes() / w.getframerate()
    sample_rate = w.getframerate()
print(f'sample rate: {sample_rate} Hz')
print(f'duration: {duration:.2f} s')
print(f'size: {os.path.getsize(wav_path) / 1024:.1f} KB')
display(Audio(wav_path))

## چک‌لیست موفقیت مرحله ۱

- [ ] نام GPU در قدم ۰ دیده شد (مثلا T4) و CUDA برابر True بود.
- [ ] قدم ۳ با return code برابر ۰ تمام شد.
- [ ] فایل step1_outputs/utterance_001.wav ساخته شد.
- [ ] صدا در قدم ۴ پخش شد و جمله قابل فهم بود.

## اگر خطا گرفتی

- خطای espeak: قدم ۱ را دوباره اجرا کن.
- خطای نصب pip: همان پیام خطا را کامل بفرست؛ مخصوصا اگر نسخه پایتون Colab برابر ۳.۱۲ باشد، نصب از روی سورس گیت‌هاب سابقه خطا دارد، پس بدون دیدن خطا، روش نصب را عوض نکن.
- اگر GPU نداده بود: در Colab از منوی Runtime گزینه Change runtime type را باز کن و GPU را روی T4 بگذار، بعد Runtime را Restart کن و از اول اجرا کن.

## مرحله بعد (هنوز اجرا نکن)

مرحله ۲: همان جمله‌ها را با steps برابر ۱، ۲، ۴، ۱۰ و ۲۵ تولید می‌کنیم و برای هر کدام RTF (زمان تقسیم بر مدت صدا) و WER با Whisper را اندازه می‌گیریم تا منحنی سرعت در برابر کیفیت رسم شود.